# 05 · TimesFM 2.5 (probe lineal y LoRA) sobre las tres entradas

Checkpoint `google/timesfm-2.5-200m-transformers` (200 M, contexto máx. 16 384, parche de 32). Una señal de 3 750 muestras cabe sin recortar:
se rellena por la izquierda con 26 ceros enmascarados hasta 3 776 = 118 parches. Cada canal pasa por el backbone; se promedian los estados ocultos
sobre los parches (1 280-d) y, con 2 canales, se concatenan antes de `Linear(C·1280, 2)`.

**Paso 1 – probe lineal:** backbone congelado; los embeddings se calculan una sola vez y se guardan en caché.
**Paso 2 – LoRA:** adaptadores r=8, α=16 en `q/k/v/o_proj` de las 20 capas; la cabeza arranca desde la del probe.

**Desviaciones del presupuesto (por costo, medido en esta GPU de 8 GB):** un paso de LoRA con 1 canal cuesta ≈ 22 ms/ejemplo, es decir ≈ 10 min por época
(≈ 20 min con 2 canales). Se entrena **3 épocas** (no 30), con bf16 autocast, batch efectivo 128 por acumulación de gradiente (micro-batch 16, u 8 con 2 canales),
y tasa de aprendizaje de los adaptadores **2e-4** en lugar de 2e-5: con B=0 al inicio y solo ≈ 640 pasos, 2e-5 casi no movería los adaptadores
(la guía del curso usa 2e-4–5e-4 para LoRA; 2e-5 es la tasa típica de fine-tuning completo). La cabeza usa 1e-3 como en el plan.

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
from bp.experiment import get_data, load_label_stats, save_val_result, device
from bp.train import fit, sanity_checks, DEFAULT_CFG
from bp.dataset import INPUT_SETS
from bp.paths import PREPROCESSED, ARTIFACTS
y_mean, y_std = load_label_stats()
dev = device()
LORA_CFG = {**DEFAULT_CFG, 'lr': 1e-3, 'head_lr': 1e-3, 'backbone_lr': 2e-4, 'epochs': 3, 'patience': 5, 'amp': True, 'eval_batch_size': 32}

## 1. Chequeos previos (TimesFM + LoRA, PPG)
Sobreajuste de 16 ejemplos (64 secuencias no caben con gradientes en 8 GB); MSE inicial sobre 256 ejemplos.

In [3]:
tr = get_data('train', ['ppg'])
r = sanity_checks('build_timesfm', tr, 1, LORA_CFG, dev, model_kw={'mode': 'lora'}, n_overfit=16, n_init=256, eval_bs=32, amp=True, overfit_steps=200)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k != 'grad_zero_layers'}, "| sin gradiente:", r['grad_zero_layers'][:3])
assert 0.8 <= r['init_mse'] <= 1.2 and r['grad_all_finite'] and r['eval_deterministic']
assert r['overfit_loss'] < 0.05, r
(ARTIFACTS / 'sanity_timesfm.json').write_text(json.dumps(r, indent=1))
del tr; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'init_mse': 0.993, 'grad_all_finite': True, 'grad_n_layers': 162, 'overfit_loss': 0.0446, 'overfit_steps': 21, 'eval_deterministic': True} | sin gradiente: []


## 2. Embeddings congelados (una sola pasada por modalidad)

In [4]:
from bp.models import TimesFMReg
backbone = TimesFMReg(1, mode='probe').to(dev).eval()
for mod in ('ppg', 'ecg'):
    out = PREPROCESSED / f'timesfm_emb_{mod}.npy'
    if out.exists():
        print(mod, 'ya calculado'); continue
    X = np.load(PREPROCESSED / f'{mod}.npy', mmap_mode='r')
    emb = np.empty((len(X), 1280), np.float32)
    t0 = time.time()
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for i in range(0, len(X), 64):
            xb = torch.from_numpy(np.ascontiguousarray(X[i:i + 64])).to(dev)[:, None, :]
            emb[i:i + 64] = backbone.embed(xb).float().cpu().numpy()
    assert np.isfinite(emb).all()
    np.save(out, emb)
    print(mod, emb.shape, f'{time.time() - t0:.0f}s', 'media/std', emb.mean().round(3), emb.std().round(3))
del backbone; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ppg (45690, 1280) 410s media/std 0.008 0.254


ecg (45690, 1280) 411s media/std 0.003 0.292


## 3. Paso 1 — probe lineal sobre embeddings
Mismo presupuesto que MLP/GRU (AdamW 1e-3, wd 1e-2, hasta 30 épocas, paciencia 5); batch 256.

In [5]:
PROBE_CFG = {**DEFAULT_CFG, 'batch_size': 256}
probes = {}
for name, inputs in INPUT_SETS.items():
    tr, va = get_data('train', inputs, embeddings=True), get_data('val', inputs, embeddings=True)
    res = fit('build_probe_head', inputs, tr, va, y_mean, y_std, cfg=PROBE_CFG, run_name='timesfm_probe_' + name)
    save_val_result(res); probes[name] = res
    print(f"==> probe {name}: val_loss {res['best_val_loss']:.4f} MAE {res['val_mae_pas']:.2f}/{res['val_mae_pad']:.2f}")
    del tr, va; torch.cuda.empty_cache()

2026/11/04 01:50:16 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\USUARIO\.conda\envs\dl_bp\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[timesfm_probe_ppg] ep 00 train 0.9699 val 0.8703 MAE 13.78/8.82 gn 0.84 t 1s


[timesfm_probe_ppg] ep 01 train 0.9183 val 0.8419 MAE 13.65/8.67 gn 0.88 t 1s


[timesfm_probe_ppg] ep 02 train 0.8981 val 0.8356 MAE 13.66/8.62 gn 0.86 t 2s


[timesfm_probe_ppg] ep 03 train 0.8854 val 0.8310 MAE 13.48/8.56 gn 0.76 t 2s


[timesfm_probe_ppg] ep 04 train 0.8768 val 0.8251 MAE 13.49/8.56 gn 0.83 t 3s


... (58 outputs left out)

[timesfm_probe_ppg+ecg] ep 19 train 0.7941 val 0.8084 MAE 13.30/8.49 gn 1.11 t 12s


[timesfm_probe_ppg+ecg] ep 20 train 0.7921 val 0.8080 MAE 13.28/8.44 gn 1.02 t 12s


[timesfm_probe_ppg+ecg] ep 21 train 0.7914 val 0.8081 MAE 13.31/8.48 gn 1.07 t 13s


==> probe ppg+ecg: val_loss 0.8074 MAE 13.32/8.48


## 4. Paso 2 — fine-tuning con LoRA (cabeza inicializada con la del probe)

In [6]:
lora = {}
for name, inputs in INPUT_SETS.items():
    head = torch.load(probes[name]['checkpoint'], weights_only=False)['state']
    init_head = lambda m, h=head: m.head.load_state_dict({k.split('head.')[1]: v for k, v in h.items()})
    cfg = {**LORA_CFG, 'micro_batch': 16 if len(inputs) == 1 else 8}
    tr, va = get_data('train', inputs), get_data('val', inputs)
    res = fit('build_timesfm', inputs, tr, va, y_mean, y_std, cfg=cfg, run_name='timesfm_lora_' + name,
              model_kw={'mode': 'lora'}, post_build=init_head)
    save_val_result(res); lora[name] = res
    print(f"==> LoRA {name}: mejor época {res['best_epoch']} val_loss {res['best_val_loss']:.4f} MAE {res['val_mae_pas']:.2f}/{res['val_mae_pad']:.2f} ({res['seconds']/60:.0f} min)")
    del tr, va; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[timesfm_lora_ppg] ep 00 train 0.8317 val 0.7908 MAE 13.09/8.39 gn 1.06 t 928s


[timesfm_lora_ppg] ep 01 train 0.7824 val 0.7813 MAE 13.03/8.37 gn 1.23 t 1856s


[timesfm_lora_ppg] ep 02 train 0.7351 val 0.7766 MAE 12.97/8.32 gn 1.16 t 2774s


==> LoRA ppg: mejor época 2 val_loss 0.7766 MAE 12.97/8.32 (48 min)


... (8 outputs left out)

[timesfm_lora_ppg+ecg] ep 00 train 0.7921 val 0.8123 MAE 13.26/8.39 gn 1.63 t 1786s


[timesfm_lora_ppg+ecg] ep 01 train 0.7426 val 0.7883 MAE 13.20/8.33 gn 1.92 t 3571s


[timesfm_lora_ppg+ecg] ep 02 train 0.6730 val 0.7886 MAE 13.17/8.37 gn 1.64 t 5354s


==> LoRA ppg+ecg: mejor época 1 val_loss 0.7883 MAE 13.20/8.33 (92 min)


In [7]:
import pandas as pd
pd.DataFrame([{'entrada': n, 'paso': p, 'mejor_epoca': r['best_epoch'], 'val_loss': r['best_val_loss'], 'MAE_PAS': r['val_mae_pas'], 'MAE_PAD': r['val_mae_pad']}
              for p, d in (('probe', probes), ('LoRA', lora)) for n, r in d.items()])

,entrada,paso,mejor_epoca,val_loss,MAE_PAS,MAE_PAD
0,ppg,probe,23,0.807489,13.268014,8.518842
1,ecg,probe,7,0.892269,14.083856,9.020889
2,ppg+ecg,probe,16,0.807365,13.320751,8.480478
3,ppg,LoRA,2,0.776558,12.969742,8.322329
4,ecg,LoRA,0,0.910151,14.124603,8.967701
5,ppg+ecg,LoRA,1,0.788308,13.196936,8.334509
